In [1]:
import boto3
import pandas as pd
from imblearn.over_sampling import SMOTE
from io import BytesIO
import os
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler,OrdinalEncoder
from sklearn.pipeline import Pipeline

In [2]:
bucket = "telco-constumer-churn-066401718601-us-east-2-an"
key_train = "processed/Train.csv"
key_test = "processed/Test.csv"
key_val = "processed/Val.csv"

In [3]:
s3 = boto3.client("s3")

response_train = s3.get_object(
    Bucket=bucket,
    Key=key_train
)

response_test = s3.get_object(
    Bucket=bucket,
    Key=key_test
)

response_val = s3.get_object(
    Bucket=bucket,
    Key=key_val
)

print(response_train.keys())

dict_keys(['ResponseMetadata', 'AcceptRanges', 'LastModified', 'ContentLength', 'ETag', 'ChecksumCRC64NVME', 'ChecksumType', 'ContentType', 'ServerSideEncryption', 'Metadata', 'Body'])


In [4]:
df_train = pd.read_csv(
    BytesIO(response_train["Body"].read())
)

df_test = pd.read_csv(
    BytesIO(response_test["Body"].read())
)

df_val = pd.read_csv(
    BytesIO(response_val["Body"].read())
)

print("Filas:", df_train.shape[0])
print("Columnas:", df_train.shape[1])

df_train.head()

Filas: 4507
Columnas: 20


,Churn,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,gender
0,No,0,No,No,30,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,No,Mailed check,19.70,625.05,Male
1,No,0,Yes,Yes,23,Yes,Yes,Fiber optic,No,No,No,No,Yes,No,Month-to-month,Yes,Electronic check,83.75,1849.95,Female
2,No,1,Yes,Yes,14,Yes,Yes,Fiber optic,No,No,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,95.80,1346.3,Female
3,No,0,No,No,56,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Bank transfer (automatic),19.70,1051.9,Female
4,No,0,No,No,63,Yes,Yes,Fiber optic,Yes,Yes,Yes,No,Yes,No,Two year,No,Credit card (automatic),98.00,6218.45,Female


In [5]:
df_train["TotalCharges"] = pd.to_numeric(
    df_train["TotalCharges"],
    errors="coerce"
)

df_test["TotalCharges"] = pd.to_numeric(
    df_test["TotalCharges"],
    errors="coerce"
)

df_val["TotalCharges"] = pd.to_numeric(
    df_val["TotalCharges"],
    errors="coerce"
)
print(df_train["TotalCharges"].isna().sum())
print(df_test["TotalCharges"].isna().sum())
print(df_val["TotalCharges"].isna().sum())

6
3
2


In [6]:
df_train = df_train.dropna(subset=["TotalCharges"]).copy()
df_test = df_test.dropna(subset=["TotalCharges"]).copy()
df_val = df_val.dropna(subset=["TotalCharges"]).copy()

In [7]:
df_train["Churn"] = df_train["Churn"].map({
    "No": 0,
    "Yes": 1
})
df_test["Churn"] = df_test["Churn"].map({
    "No": 0,
    "Yes": 1
})
df_val["Churn"] = df_val["Churn"].map({
    "No": 0,
    "Yes": 1
})


In [8]:
X_train=df_train.drop(columns=["Churn"])
y_train=df_train['Churn']

X_test=df_test.drop(columns=["Churn"])
y_test=df_test['Churn']

X_val=df_val.drop(columns=["Churn"])
y_val=df_val['Churn']

In [9]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns

print("Numéricas:")
print(numeric_features.tolist())

print("\nCategóricas:")
print(categorical_features.tolist())

Numéricas:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categóricas:
['Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'gender']


In [10]:
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)
categorical_transformer = Pipeline( steps=[ ( "onehot", OneHotEncoder( handle_unknown="ignore", sparse_output=False ) ) ] )

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [12]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

In [13]:
feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)
X_val_processed = pd.DataFrame(
    X_val_processed,
    columns=feature_names,
    index=X_val.index
)

In [14]:
smote = SMOTE(random_state=42)


In [15]:
X_train_processed, y_train = smote.fit_resample(X_train_processed, y_train)

In [16]:
y_train.value_counts()

Churn
0    3293
1    3293
Name: count, dtype: int64

In [17]:
train_processed = X_train_processed.copy()
train_processed["Churn"] = y_train.values

test_processed = X_test_processed.copy()
test_processed["Churn"] = y_test.values

val_processed = X_val_processed.copy()
val_processed["Churn"] = y_val.values

In [18]:
print(train_processed["Churn"].value_counts())
print(train_processed["Churn"].unique())

Churn
0    3293
1    3293
Name: count, dtype: int64
[0 1]


In [19]:
def move_target_first(df, target="Churn"):
    columns = [target] + [col for col in df.columns if col != target]
    return df[columns]

In [20]:
train_processed = move_target_first(train_processed)
val_processed = move_target_first(val_processed)
test_processed = move_target_first(test_processed)

In [21]:
os.makedirs("data/processed", exist_ok=True)

In [22]:
train_processed.to_csv(
    "data/processed/train.csv",
    index=False
)

test_processed.to_csv(
    "data/processed/test.csv",
    index=False
)

val_processed.to_csv(
    "data/processed/val.csv",
    index=False
)

In [23]:
print(os.listdir("data/processed"))

['train.csv', 'test.csv', 'val.csv']


In [24]:
s3.upload_file(
    "data/processed/train.csv",
    bucket,
    "Processed/train.csv"
)

s3.upload_file(
    "data/processed/val.csv",
    bucket,
    "Processed/val.csv"
)

s3.upload_file(
    "data/processed/test.csv",
    bucket,
    "Processed/test.csv"
)

print("✅ Archivos subidos a S3")

✅ Archivos subidos a S3


In [25]:
response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix="Processed/"
)

for obj in response.get("Contents", []):
    print(obj["Key"], obj["Size"], "bytes")

Processed/test.csv 344755 bytes
Processed/train.csv 1707506 bytes
Processed/val.csv 275916 bytes


In [26]:
train_s3 = f"s3://{bucket}/Processed/train.csv"
validation_s3 = f"s3://{bucket}/Processed/validation.csv"
test_s3 = f"s3://{bucket}/Processed/test.csv"

print(train_s3)
print(validation_s3)
print(test_s3)

s3://telco-constumer-churn-066401718601-us-east-2-an/Processed/train.csv
s3://telco-constumer-churn-066401718601-us-east-2-an/Processed/validation.csv
s3://telco-constumer-churn-066401718601-us-east-2-an/Processed/test.csv
